In [1]:
!pip3 install --upgrade boto3   

> **Actualizado en noviembre 2026.** Amazon Titan Text y Anthropic Claude 3.x llegaron a su fin de vida en Amazon Bedrock (ver [Model lifecycle](https://docs.aws.amazon.com/bedrock/latest/userguide/model-lifecycle.html)). Este notebook usa ahora [Claude Haiku 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-haiku-5-5.html) a través del perfil de inferencia global. Claude 4.7+ ya no acepta `temperature`, `top_p` ni `top_k` ([docs de Anthropic](https://platform.claude.com/docs/en/models/haiku-5-5/overview)), por eso `inferenceConfig` solo lleva `maxTokens`. Claude 5.5 tiene [adaptive thinking](https://platform.claude.com/docs/en/build-with-claude/thinking) activado por defecto: la respuesta puede empezar con un bloque `reasoningContent` antes del bloque `text`, por eso el código toma los bloques de texto y no `content[0]`. Como estos ejemplos piden respuestas cortas con pocos tokens, se desactiva el [adaptive thinking](https://platform.claude.com/docs/en/build-with-claude/thinking) con `additionalModelRequestFields={"thinking": {"type": "disabled"}}`; si no, el modelo puede gastar todo `maxTokens` pensando y no devolver texto.

# Técnicas básicas de prompt

In [2]:
import boto3
import json


In [3]:
bedrock_runtime = boto3.client(
    service_name='bedrock-runtime', 
    region_name='us-east-1'
)

In [4]:
def api_bedrock(prompt):
    response = bedrock_runtime.converse(
    modelId="global.anthropic.claude-haiku-5-5",
    messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "text": prompt
                        }
                    ]
                }
            ],
    # Claude 4.7+ no acepta temperature ni topP: https://platform.claude.com/docs/en/models/haiku-5-5/overview
    inferenceConfig = {
          "maxTokens":250
          },
    # https://platform.claude.com/docs/en/build-with-claude/thinking
    additionalModelRequestFields = {"thinking": {"type": "disabled"}}
          )

    try:
        result = "".join(block["text"] for block in response['output']['message']['content'] if "text" in block) \
        + '\n--- Latency: ' + str(response['metrics']['latencyMs']) \
        + 'ms - Input tokens:' + str(response['usage']['inputTokens']) \
        + ' - Output tokens:' + str(response['usage']['outputTokens']) + ' ---\n'
        return result
    except Exception as e:
        print(e)
        result = "Output parsing error"
    print(result)
    return result

## Zero-shot prompting


In [5]:
publicacion = "¡No te pierdas la revolución del vehículo eléctrico! AnyCompany está abandonando los potentes coches por vehículos eléctricos, lo que crea una gran oportunidad para los inversores."

In [6]:
prompt1= "Cuénteme el sentimiento de la siguiente publicación en las redes sociales y clasifíquelo como positivo, negativo o neutral: "+publicacion
api_bedrock(prompt1)

'**Clasificación: Positivo**\n\n**Análisis del sentimiento:**\n\n- **Tono entusiasta:** La expresión "¡No te pierdas la revolución del vehículo eléctrico!" transmite emoción y urgencia.\n- **Lenguaje de oportunidad:** Frases como "gran oportunidad para los inversores" presentan la noticia de forma favorable y atractiva.\n- **Enfoque en el progreso:** Describir el cambio como una "revolución" sugiere avance e innovación.\n- **Ausencia de matices negativos:** No se mencionan riesgos, pérdidas ni consecuencias desfavorables. El único elemento de "pérdida" aparece en "no te pierdas", que funciona como llam\n--- Latency: 2481ms - Input tokens:126 - Output tokens:250 ---\n'

## Few-shot prompt

In [7]:
publicacion = "Manufacturing plant is the latest target in investigation by state officials."

In [8]:
prompt1=f"""Tell me the sentiment of the following headline and categorize it as either positive, negative, or neutral. 
Here are some examples:
Research firm fends off allegations of impropriety over new technology.
Answer: Negative
Offshore windfarms continue to thrive as vocal minority in opposition dwindles.
Answer: Positive
{publicacion}
Answer:"""
api_bedrock(prompt1)

'Negative\n--- Latency: 1257ms - Input tokens:142 - Output tokens:5 ---\n'

## Chain-of-thought prompting


In [9]:
prompt = """¿Qué vehículo requiere un pago inicial mayor según la siguiente información? 

El costo total del vehículo A es de $40 000 y requiere un pago inicial del 30 por ciento. 

El costo total del vehículo B es de $50 000 y requiere un pago inicial del 20 por ciento. 

(Piensa paso a paso)
"""

In [10]:
api_bedrock(prompt)

'Calculemos el pago inicial de cada vehículo paso a paso.\n\n**Vehículo A:**\n- Costo total: $40 000\n- Pago inicial: 30 % de $40 000\n- 0.30 × $40 000 = **$12 000**\n\n**Vehículo B:**\n- Costo total: $50 000\n- Pago inicial: 20 % de $50 000\n- 0.20 × $50 000 = **$10 000**\n\n**Comparación:**\n$12 000 (Vehículo A) > $10 000 (Vehículo B)\n\n**Respuesta:** El **Vehículo A** requiere un pago inicial mayor, de $12 000, frente a los $10 000 del Vehículo B. Aunque el Vehículo B cuesta más en total\n--- Latency: 2036ms - Input tokens:124 - Output tokens:250 ---\n'

## Chain-of-thought prompt ejemplo: Few-shot

In [11]:
prompt = """En una semana determinada, los espectadores de un canal de televisión son los siguientes: 
Lunes: 6.500 espectadores 
Martes: 6.400 espectadores 
Miércoles: 6.300 espectadores 
Pregunta: ¿Cuántos espectadores podemos esperar el viernes? 
Respuesta: Según los números dados y sin más información, hay una disminución diaria de 100 espectadores. 
Si asumimos que esta tendencia continuará durante los siguientes días, podemos esperar 6.200 espectadores el día siguiente que sería jueves, y por tanto 6.100 espectadores el día siguiente que sería viernes. 
Pregunta: ¿Cuántos espectadores podemos esperar el sábado? 
(Piense paso a paso) Respuesta:
"""

In [12]:
api_bedrock(prompt)

'Siguiendo la misma tendencia, la disminución diaria es de 100 espectadores:\n\n- Lunes: 6.500\n- Martes: 6.400\n- Miércoles: 6.300\n- Jueves: 6.200\n- Viernes: 6.100\n- Sábado: 6.000\n\nPor tanto, podemos esperar **6.000 espectadores el sábado**, asumiendo que la tendencia de descenso de 100 espectadores diarios se mantiene.\n--- Latency: 1361ms - Input tokens:282 - Output tokens:159 ---\n'